In [ ]:
import polars as pl
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import spacy 
nlp = spacy.load("pt_core_news_lg")

In [ ]:
textos = pl.read_parquet('../database/textos/*.parquet').select('id', 'texto')
metadata = pl.read_parquet('../database/metadata/*.parquet').select('id', 'pubName', 'name', 'artType', 'pubDate', 'artCategory', 'pdfPage')
df = textos.join(metadata, on='id', how='inner')
del textos, metadata

# 1. Entendimento do acervo

## Volume de documentos por período

In [ ]:
volume_por_ano = df.group_by(pl.col('pubDate').str.to_datetime().dt.strftime('%Y/%m'), pl.col('pubName').str.head(3)).agg(pl.len()).sort('pubDate', 'pubName')
fig = px.area(volume_por_ano.to_pandas(), x='pubDate', y='len', color='pubName', title='Volume de artigos por mês e publicação')
fig.show()

## Distribuição por fonte, autor, órgão ou categoria

## Tamanho dos textos

In [ ]:
tamanho_por_publicacao = (
    df.select(
        pl.col("pubName").str.head(3).alias("pubName"),
        pl.col("texto").str.len_chars().alias("tamanho")
    )
    .drop_nulls("tamanho")
)

tamanho_pd = tamanho_por_publicacao.to_pandas()

limite_superior = tamanho_pd["tamanho"].quantile(0.95)
plot_data = tamanho_pd[tamanho_pd["tamanho"] <= limite_superior]

plt.figure(figsize=(12, 6))
sns.histplot(
    data=plot_data,
    x="tamanho",
    hue="pubName",
    bins=240,
    stat="count",
    common_norm=False,
    element="step",
    fill=False,
)
plt.title("Distribuicao da contagem de caracteres por texto (até p95)")
plt.xlabel("Quantidade de caracteres")
plt.ylabel("Contagem de textos")
plt.tight_layout()
plt.show()


In [ ]:
tamanho_por_publicacao.describe()

In [ ]:
df.filter(pl.col('texto').str.len_chars() == 0)

In [ ]:
df.filter(pl.col('texto').str.len_chars() > 1e6).sort(pl.col('texto').str.len_chars(), descending=True).show(limit=-1)

In [ ]:
df.filter(pl.col('artCategory').str.to_lowercase().str.contains('ministério da defesa')).filter(pl.col('texto').str.len_chars() > 1e6).sort(pl.col('texto').str.len_chars(), descending=True).show(limit=-1)

In [ ]:
print(df.filter(pl.col('id')=='120040826266').item(0,'texto'))

## Percentagem de textos duplicados ou quase duplicados


In [ ]:
df.filter(df.select('texto').is_duplicated())

## Percentual de textos vazios ou muito curtos

In [ ]:
df.filter(pl.col('texto').str.len_chars() < 140).show(limit=-1, fmt_str_lengths=150)

# 2. Qualidade e padronização textual

## Sinônimos com grafias diferentes

In [ ]:
doc = nlp(df.item(0, 'texto'))

palavras_relevantes = [(token.text, token.pos_) for token in doc if token.has_vector and not token.is_stop and not token.is_punct]

word = nlp.vocab['permite']

queries = [w for w in nlp.vocab if w.has_vector and w.is_lower and w.lower_ != word.text]
by_similarity = sorted(queries, key=lambda w: word.similarity(w), reverse=True)

print([w.text for w in by_similarity])

## Siglas não padronizadas

## Nomes próprios com múltiplas grafias

## Mudanças históricas de vocabulário

## Boilerplates (cabeçalhos e rodapés)

# 3. Descoberta de temas

## Assuntos dos textos

## Temas que dominam o acervo


## Mudança dos temas ao longo do tempo


## Temas exclusivos a certas organizações

# 4. Entidades e relações

## Pessoas

## Organizações


## Localidades


## Cargos


## Valores monetários

## Datas


## Objetos contratuais

## Normas e referências legais

# 5. Evolução temporal

## Variações de frequências de termos

## Eventos que alteram o vocabulário institucional

## Mudança do estilo redacional

## Temas cíclicos


# 6. Padrões de linguagem

## Textos formais ou padronizados


## Verbosidade dos documentos

## Uso de e linguagem técnica administrativa

## Níveis de repetição

## Densidade de termos jurídicos, financeiros ou operacionais

# 7. Anomalias e casos raros

## Documentos com vocabulário muito incomum

## Entidades que aparecem poucas vezes, porém em textos relevantes

## Textos que podem pertencer à classe errada

## Mudanças abruptas de temas em determinados períodos

## Contratos, decisões ou atos com redação única

# 8. Classificação e taxonomia

## Tema principal

## Subtipo documental

## Área funcional

## Natureza do ato

## Tipo de decisão

## Impacto institucional

# 9. Indicadores derivados de texto

## Índice de complexidade textual

## Índice de repetição

## Proporção de termos de contratação, pessoal, norma, logística

## Quantidade de entidades por documento

## Densidade de valores monetários por documento

## Presença de termos chave como urgência, dispensa, nomeação, exoneração, aquisição

# 10. Perguntas do KDD

## Quais são os grandes blocos temáticos do acervo?

## Como esses blocos temáticos tem mudado ao longo dos anos?

## Quais entidades mais aparecem e em quais contextos?

## Existem padrões distintos entre grupos, fontes ou instituições?

## Quais documentos fogem ao padrão esperado?

## Quais termos indicam muidanças de política, prioridade ou atividade?

## Que relações entre entidades são recorrentes?

## Quais temas estão crescendo ou desaparecendo?

## Quais documentos são altamente similares entre si?

## Há sinais de mudanças de linguagem institucional ao longo dos anos?